# Sofifa Scraping

In [5]:
import csv
import time
import random
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright
import sys
import asyncio
import os
from playwright_stealth import Stealth

# Force Windows to use the Proactor Event Loop for subprocess support
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# REPLACE THIS with your massive URL containing all 46 columns
BASE_URL = "https://sofifa.com/players?&showCol%5B%5D=pi&showCol%5B%5D=ae&showCol%5B%5D=by&showCol%5B%5D=hi&showCol%5B%5D=pf&showCol%5B%5D=oa&showCol%5B%5D=pt&showCol%5B%5D=bp&showCol%5B%5D=gu&showCol%5B%5D=vl&showCol%5B%5D=wg&showCol%5B%5D=ta&showCol%5B%5D=cr&showCol%5B%5D=fi&showCol%5B%5D=he&showCol%5B%5D=sh&showCol%5B%5D=vo&showCol%5B%5D=ts&showCol%5B%5D=dr&showCol%5B%5D=cu&showCol%5B%5D=fr&showCol%5B%5D=lo&showCol%5B%5D=bl&showCol%5B%5D=to&showCol%5B%5D=ac&showCol%5B%5D=sp&showCol%5B%5D=ag&showCol%5B%5D=tp&showCol%5B%5D=so&showCol%5B%5D=ju&showCol%5B%5D=st&showCol%5B%5D=sr&showCol%5B%5D=ln&showCol%5B%5D=te&showCol%5B%5D=vi&showCol%5B%5D=pe&showCol%5B%5D=td&showCol%5B%5D=ma&showCol%5B%5D=sa&showCol%5B%5D=sl&showCol%5B%5D=tg&showCol%5B%5D=gd&showCol%5B%5D=gh&showCol%5B%5D=gc&showCol%5B%5D=gp&showCol%5B%5D=gr"
CSV_FILENAME = "data/sofifa/newdata/sofifa_players.csv"
TOTAL_PLAYERS = 400000 
PLAYERS_PER_PAGE = 60

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        # SoFifa's first column (the avatar picture) has no text in the header.
        # We dynamically rename this header to "ID" for our CSV.
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    return headers

def extract_rows_from_html(soup, limit=None):
    player_data = []
    rows = soup.select("table tbody tr")
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker (ads have very few columns)
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION (Strictly Isolated)
                if 'col-name' in classes:
                    links = td.find_all("a", href=lambda h: h and "/player/" in h)
                    name_text = ""
                    for a in links:
                        # Find the hyperlink that actually has the text of the name, not the avatar image
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        # Fallback just in case
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/player/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER STATS (Including the real ID column)
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
                    
            player_data.append(row_values)
            
            if limit and len(player_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return player_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    os.makedirs(os.path.dirname(CSV_FILENAME), exist_ok=True)

    with sync_playwright() as p:
        # 1. Setup Persistent Profile Folder
        profile_path = os.path.join(os.getcwd(), "data", "sofifa_profile")
        os.makedirs(profile_path, exist_ok=True)
        
        # 2. Launch actual Chrome with saved cookies
        context = p.chromium.launch_persistent_context(
            user_data_dir=profile_path,
            channel="chrome", 
            headless=False,
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        
        # 3. Resource Blocker
        def block_heavy_resources(route):
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked for extreme speed).\n")

        # 4. Grab the default tab and apply Stealth
        page = context.pages[0]
        stealth = Stealth()
        stealth.apply_stealth_sync(page)
        
        print("Launching persistent stealth browser to solve Cloudflare challenge...")
        
        try:
            # 3. Now it is safe to navigate
            page.goto(f"{BASE_URL}&offset=0")
            page.wait_for_selector("table tbody tr", timeout=30000)
            
            # Dynamic wait
            page.wait_for_function(
                "() => document.querySelectorAll('table tbody tr td').length > 100",
                timeout=15000
            )
            print("Challenge passed! Table loaded.")
            
        except Exception as e:
            print(f"Failed to bypass Cloudflare. Error: {e}")
            context.close()
            return

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-PLAYER VALIDATION TEST ---")
            
            # # Reload page with Turbo Mode active
            # page.goto(f"{BASE_URL}&offset=0")
            # page.wait_for_selector("table tbody tr") 
            
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            players = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, player in enumerate(players):
                player_dict = dict(zip(columns, player))
                print(f"Player {i+1}: {player_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            # 1. Check if we have an existing file to resume from
            start_offset = 0
            if os.path.exists(CSV_FILENAME):
                with open(CSV_FILENAME, "r", encoding="utf-8") as f:
                    # Count lines minus 1 for the header
                    existing_rows = sum(1 for line in f) - 1
                    if existing_rows > 0:
                        # Round down to the nearest multiple of 60 to ensure clean pagination
                        start_offset = (existing_rows // PLAYERS_PER_PAGE) * PLAYERS_PER_PAGE

            print(f"--- STARTING PRODUCTION SCRAPE ---")
            if start_offset > 0:
                print(f"[Resume Mode] Found {existing_rows} existing players. Resuming from offset {start_offset}...\n")
            else:
                print("[New Run] No existing data found. Starting from scratch...\n")

                # Initialize fresh CSV file with headers
                page.goto(f"{BASE_URL}&offset=0")
                page.wait_for_selector("table tbody tr")
                soup = BeautifulSoup(page.content(), "html.parser")
                headers = extract_headers_from_html(soup)
                
                with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                    writer = csv.writer(file)
                    writer.writerow(headers)
                
            # 2. Master Loop (Starts at start_offset)
            with tqdm(total=TOTAL_PLAYERS, initial=start_offset, desc="Scraping SoFifa", unit=" players") as pbar:
                for offset in range(start_offset, TOTAL_PLAYERS, PLAYERS_PER_PAGE):
                    url = f"{BASE_URL}&offset={offset}"
                    
                    success = False
                    for attempt in range(3):
                        try:
                            # Only navigate if it's not the first load (or if we are resuming)
                            if offset != 0 or start_offset > 0 or attempt > 0:
                                page.goto(url)
                                page.wait_for_selector("table tbody tr", timeout=30000)
                                
                                # Dynamic wait: Checks if the whole table has populated
                                page.wait_for_function(
                                    "() => document.querySelectorAll('table tbody tr td').length > 100",
                                    timeout=15000
                                )
                                
                            soup = BeautifulSoup(page.content(), "html.parser")
                            players = extract_rows_from_html(soup)
                            
                            # Append strictly to CSV (mode="a")
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(players)
                            
                            pbar.update(len(players))
                            success = True
                            
                            if len(players) == 0:
                                print("\n[Notice] No more players found. Database exhausted.")
                                context.close()
                                return
                            break 
                            
                        except Exception as e:
                            print(f"\n[Error on offset {offset}]. Attempt {attempt + 1}/3.")
                            print(f"Details: {str(e)}")  # <--- This will tell us exactly what failed
                            time.sleep(5)
                    
                    if not success:
                        print(f"\n[Fatal] Failed to fetch offset {offset} after 3 attempts. Stopping script to prevent data gaps.")
                        break

                    # Polite delay
                    time.sleep(random.uniform(1.5, 3.0))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        # Safely shut down Chromium
        context.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_17536\2740390964.py:15: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_17536\2740390964.py:15: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [6]:
run_test()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- RUNNING 10-PLAYER VALIDATION TEST ---
Successfully Fetched! Extracted 50 Columns.
--------------------------------------------------
Player 1: {'Unknown': '', 'Name': 'K. Coulibaly CB CDM CM', 'Age': '18', 'Overall rating': '71 +2', 'Potential': '85 +3', 'Team & Contract': 'SV Werder Bremen 2025 ~ 2029', 'ID': '77636', 'Birth year': '2007', 'Height': '191cm 6\'3"', 'foot': 'Left', 'Best position': 'CB', 'Growth': '14', 'Value': '€4.1M', 'Wage': '€11K', 'Total attacking': '248', 'Crossing': '45 +2', 'Finishing': '35 +2', 'Heading accuracy': '65 +3', 'Short passing': '70 +2', 'Volleys': '33', 'Total skill': '238', 'Dribbling': '50 +3', 'Curve': '38 +4', 'FK Accuracy': '25', 'Long passing': '66 +3', 'Ball control': '59 +1', 'Total movement': '322', 'Acceleration': '61', 'Sprint speed': '68', 'Agility': '62', 'Total power': 

In [ ]:
run_production()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- STARTING PRODUCTION SCRAPE ---
[Resume Mode] Found 975 existing players. Resuming from offset 960...



Scraping SoFifa:   0%|          | 970/400000 [00:22<253:14:27,  2.28s/ players]

In [12]:
import csv
import time
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# Exact URL, no offset parameters needed for leagues
BASE_URL = "https://sofifa.com/leagues"
CSV_FILENAME = "data/sofifa/newdata/sofifa_raw_leagues.csv"

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers():
    return ["sofifa_id", "sofifa_name", "sofifa_country"]

def extract_rows_from_html(soup, limit=None):
    league_data = []
    rows = soup.select("table tbody tr")
    
    if limit:
        rows = rows[:limit]
        
    for row in tqdm(rows, desc="Parsing Leagues", unit=" league"):
        try:
            cols = row.find_all("td")
            
            # Bulletproof check
            if len(cols) < 3:
                continue
                
            # 1. ID & NAME EXTRACTION
            name_td = cols[1]
            link = name_td.find("a", href=lambda h: h and "/league/" in h)
            
            league_id = link['href'].split('/')[2] if link else ""
            league_name = link.get_text(strip=True) if link else name_td.get_text(strip=True)
            
            # 2. BULLETPROOF COUNTRY EXTRACTION
            # Scans the entire row for the nation link instead of guessing the column index
            country_name = "Unknown"
            nation_link = row.find("a", href=lambda h: h and "na=" in h)
            
            if nation_link:
                # 1st Priority: Extract from the flag image's title attribute
                img = nation_link.find("img")
                if img and img.has_attr("title"):
                    country_name = img["title"]
                # 2nd Priority: Extract from the anchor tag's title attribute
                elif nation_link.has_attr("title"):
                    country_name = nation_link["title"]
            
            league_data.append([league_id, league_name, country_name])
            
        except Exception as e:
            continue
            
    return league_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    with sync_playwright() as p:
        # headless=False is required to pass Cloudflare's bot detection
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        print("Launching browser to solve potential Cloudflare challenge...")
        page.goto(BASE_URL)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Table loaded successfully.")
        except Exception as e:
            print("Failed to load table in time. Please try again.")
            browser.close()
            return

        # === RESOURCE BLOCKER (TURBO MODE) ===
        def block_heavy_resources(route):
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked).\n")

        # Extract the page source once
        soup = BeautifulSoup(page.content(), "html.parser")
        headers = extract_headers()

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 5-LEAGUE VALIDATION TEST ---")
            leagues = extract_rows_from_html(soup, limit=5)

            print(f"\nSuccessfully Fetched! Extracted {len(headers)} Columns.")
            print("-" * 50)
            for i, league in enumerate(leagues):
                league_dict = dict(zip(headers, league))
                print(f"League {i+1}: {league_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            print("--- STARTING SINGLE-PAGE PRODUCTION SCRAPE ---")
            
            leagues = extract_rows_from_html(soup)
            
            # Write to CSV in one clean operation
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
                writer.writerows(leagues)

            print(f"\nScraping complete! {len(leagues)} leagues safely saved to {CSV_FILENAME}")

        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_34212\1149642899.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_34212\1149642899.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [13]:
run_test()

Launching browser to solve potential Cloudflare challenge...
Table loaded successfully.
Turbo mode activated (Images/CSS blocked).

--- RUNNING 5-LEAGUE VALIDATION TEST ---


Parsing Leagues: 100%|██████████| 5/5 [00:00<00:00, 6419.20 league/s]


Successfully Fetched! Extracted 3 Columns.
--------------------------------------------------
League 1: {'sofifa_id': '1', 'sofifa_name': 'Superliga', 'sofifa_country': 'Denmark'}
League 2: {'sofifa_id': '4', 'sofifa_name': 'Pro League', 'sofifa_country': 'Belgium'}
League 3: {'sofifa_id': '7', 'sofifa_name': 'Série A', 'sofifa_country': 'Brazil'}
League 4: {'sofifa_id': '10', 'sofifa_name': 'Eredivisie', 'sofifa_country': 'Netherlands'}
League 5: {'sofifa_id': '13', 'sofifa_name': 'Premier League', 'sofifa_country': 'England'}
--------------------------------------------------
Test complete. Clear to run production scrape.



In [14]:
run_production()

Launching browser to solve potential Cloudflare challenge...
Table loaded successfully.
Turbo mode activated (Images/CSS blocked).

--- STARTING SINGLE-PAGE PRODUCTION SCRAPE ---


Parsing Leagues: 100%|██████████| 52/52 [00:00<00:00, 8702.22 league/s]


Scraping complete! 52 leagues safely saved to data/sofifa/newdata/sofifa_raw_leagues.csv


In [21]:
import os
import re
import pandas as pd
import unicodedata
from rapidfuzz import fuzz

# ==========================================
# 1. CONFIGURATION & SEMANTIC MAPPING
# ==========================================
PATHS = {
    "matched": "newdata/leagues/matched",
    "partial": "newdata/leagues/partial match",
    "unmatched": "newdata/leagues/no match"
}
for path in PATHS.values():
    os.makedirs(path, exist_ok=True)

# Semantic dictionary to resolve geographic database inconsistencies
COUNTRY_MAP = {
    'england': 'united kingdom',
    'scotland': 'united kingdom',
    'wales': 'united kingdom',
    'northern ireland': 'united kingdom',
    'usa': 'united states',
    'korea republic': 'south korea',
    'republic of ireland': 'ireland',
    'china pr': 'china',
    'turkiye': 'turkey' 
}

def normalize_country(country_str):
    """Strips accents from country names and applies semantic mapping."""
    if pd.isna(country_str): return ""
    
    c = str(country_str).lower()
    c = unicodedata.normalize('NFKD', c).encode('ASCII', 'ignore').decode('utf-8')
    c = c.strip()
    
    return COUNTRY_MAP.get(c, c)

def clean_league_name(name):
    """Strips accents, punctuation, and standardizes text for the math engine."""
    if pd.isna(name): return ""
    name = str(name).lower()
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name = re.sub(r'[^a-z0-9\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# ==========================================
# 2. LOAD & PREP DATASETS
# ==========================================
master_df = pd.read_csv('data/unified_tables/leagues/matched/final_combined_leagues.csv')
sofifa_df = pd.read_csv('data/sofifa/newdata/sofifa_raw_leagues.csv')
# Enforce the sofifa_ prefix for column clarity downstream
sofifa_df = sofifa_df.rename(columns=lambda x: x if x.startswith('sofifa_') else f'sofifa_{x}')

# Dimensionality Reduction (Squash time-series)
unique_base_leagues = master_df[['soccersolver_id', 'soccersolver_name', 'soccersolver_country']].drop_duplicates()

# ==========================================
# 3. TRIAGE MATCHING ENGINE (DUAL-SCORING)
# ==========================================
potential_matches = []

for _, base_row in unique_base_leagues.iterrows():
    best_score = 0
    best_match_id = None
    
    base_country_norm = normalize_country(base_row['soccersolver_country'])
    base_name_clean = clean_league_name(base_row['soccersolver_name'])
    
    for _, sof_row in sofifa_df.iterrows():
        sof_country_norm = normalize_country(sof_row['sofifa_country'])
        
        # Geographic Anchoring (Fast-Fail)
        if base_country_norm != sof_country_norm:
            continue
            
        sof_name_clean = clean_league_name(sof_row['sofifa_name'])
        
        # DUAL-SCORING ENGINE
        # Score 1: Token Sort (Handles inverted structures: "League Premier" vs "Premier League")
        score_token = fuzz.token_sort_ratio(base_name_clean, sof_name_clean)
        
        # Score 2: Stripped Ratio (Handles spacing inconsistencies: "La Liga 2" vs "LaLiga 2")
        score_stripped = fuzz.ratio(base_name_clean.replace(" ", ""), sof_name_clean.replace(" ", ""))
        
        # Take the absolute best score of the two methods
        score = max(score_token, score_stripped)
        
        if score > best_score:
            best_score = score
            best_match_id = sof_row['sofifa_id']
            
    # Capture everything 40% and above into the global pool
    if best_match_id and best_score >= 40:
        potential_matches.append({
            'soccersolver_id': base_row['soccersolver_id'],
            'sofifa_id': best_match_id,
            'sofifa_match_score': best_score
        })

pool_df = pd.DataFrame(potential_matches)

# ==========================================
# 4. GLOBAL DEDUPLICATION (1-TO-1 ENFORCER)
# ==========================================
if not pool_df.empty:
    pool_df = pool_df.sort_values('sofifa_match_score', ascending=False)
    pool_df = pool_df.drop_duplicates(subset=['soccersolver_id'])
    pool_df = pool_df.drop_duplicates(subset=['sofifa_id'])

# ==========================================
# 5. HISTORICAL EXPLOSION (PRE-ROUTING MERGE)
# ==========================================
# Merge the 1-to-1 dictionary onto the massive time-series master file
exploded_df = master_df.merge(pool_df, on='soccersolver_id', how='left')
# Pull in the rest of the Sofifa columns
final_df = exploded_df.merge(sofifa_df, on='sofifa_id', how='left')

# ==========================================
# 6. ZERO DATA LOSS ROUTING
# ==========================================
# 1. Perfect Matches (> 75)
perfect_df = final_df[final_df['sofifa_match_score'] > 75].copy()

# 2. Partial Matches (40 to 75)
partial_df = final_df[(final_df['sofifa_match_score'] >= 40) & (final_df['sofifa_match_score'] <= 75)].copy()
partial_df['APPROVED'] = '' 

# 3. SoccerSolver Orphans (Failed to hit 40% threshold)
base_orphans = final_df[final_df['sofifa_match_score'].isna()].copy()
sofifa_cols = [c for c in final_df.columns if c.startswith('sofifa_')]
base_orphans.drop(columns=sofifa_cols, errors='ignore', inplace=True)

# 4. Sofifa Orphans (Using index isolation)
matched_sofifa_ids = set(pool_df['sofifa_id'].dropna()) if not pool_df.empty else set()
sofifa_orphans = sofifa_df[~sofifa_df['sofifa_id'].isin(matched_sofifa_ids)].copy()

# ==========================================
# 7. EXPORTS & MATH VERIFICATION
# ==========================================
perfect_df.to_csv(os.path.join(PATHS['matched'], 'leagues_matched.csv'), index=False)
partial_df.to_csv(os.path.join(PATHS['partial'], 'leagues_partial.csv'), index=False)
base_orphans.to_csv(os.path.join(PATHS['unmatched'], 'soccersolver_leagues_unmatched.csv'), index=False)
sofifa_orphans.to_csv(os.path.join(PATHS['unmatched'], 'sofifa_leagues_unmatched.csv'), index=False)

print("\n--- Routing Complete ---")
print(f"- Perfect Matches (Rows): {len(perfect_df)}")
print(f"- Require Manual Review (Rows): {len(partial_df)}")
print(f"- Unmatched SoccerSolver (Rows): {len(base_orphans)}")
print(f"- Unmatched Sofifa (Entities): {len(sofifa_orphans)}")

total_ss_processed = len(perfect_df) + len(partial_df) + len(base_orphans)
print(f"\n[MATH CHECK] Perfect + Partial + Orphans = {total_ss_processed} (Should exactly equal {len(master_df)})")


--- Routing Complete ---
- Perfect Matches (Rows): 162
- Require Manual Review (Rows): 28
- Unmatched SoccerSolver (Rows): 77
- Unmatched Sofifa (Entities): 24

[MATH CHECK] Perfect + Partial + Orphans = 267 (Should exactly equal 267)


In [22]:
# ==========================================
# CONFIGURATION
# ==========================================
PATH_MATCHED = 'newdata/leagues/matched/leagues_matched.csv'
PATH_PARTIAL = 'newdata/leagues/partial match/leagues_partial.csv' 
PATH_FINAL_OUTPUT = 'newdata/leagues/matched/master_leagues_final_integrated.csv'

# ==========================================
# 1. LOAD DATASETS
# ==========================================
try:
    perfect_df = pd.read_csv(PATH_MATCHED)
    partial_df = pd.read_csv(PATH_PARTIAL)
except FileNotFoundError as e:
    print(f"Error: {e}. Ensure you have run the matching script and reviewed the partials.")
    raise

# ==========================================
# 2. ISOLATE APPROVED PARTIALS
# ==========================================
# Safely convert the APPROVED column to numeric, ignoring text or blanks
partial_df['APPROVED'] = pd.to_numeric(partial_df['APPROVED'], errors='coerce')

# Isolate explicitly approved leagues
approved_partials = partial_df[partial_df['APPROVED'] == 1].copy()

# Drop the helper column to align schemas perfectly
approved_partials.drop(columns=['APPROVED'], errors='ignore', inplace=True)

# ==========================================
# 3. INTEGRATION & EXPORT
# ==========================================
# Concatenate vertically
final_master_df = pd.concat([perfect_df, approved_partials], ignore_index=True)

# Export the final master file
final_master_df.to_csv(PATH_FINAL_OUTPUT, index=False)

# ==========================================
# 4. VERIFICATION
# ==========================================
print("--- Post-Review Integration Complete ---")
print(f"Perfect Matches Loaded:    {len(perfect_df)}")
print(f"Approved Partials Added:   {len(approved_partials)}")
print(f"Total Leagues in Master:   {len(final_master_df)}")
print(f"\nFinal dataset saved to: {PATH_FINAL_OUTPUT}")

--- Post-Review Integration Complete ---
Perfect Matches Loaded:    162
Approved Partials Added:   7
Total Leagues in Master:   169

Final dataset saved to: newdata/leagues/matched/master_leagues_final_integrated.csv
